# NB38: Categorical Niche Breadth

Extends NB37 with: (A) KO subcategory PGLS, (B) soil habitat categorical Levins B, (C) Spark-based categorical layers (ESA CCI, Köppen-Geiger, population density, SoilGrids CEC/clay, GEMAS+NGSA), (D) biome-stratified PGLS with genome isolation source restriction, (F) external datasets (EPA Ecoregions, GLWD wetland, NADP, N deposition), (G) summary figures.

**Off-cluster cells:** Setup, A, B, C6, C4b, D3, F, G — run immediately.
**Spark cells:** C0–C5, D1, D2 — run on JupyterHub cluster.

## Setup

In [1]:
import os, sys, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
warnings.filterwarnings("ignore")

PROJECT = Path("/home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology")
DATA    = PROJECT / "data"
FIGS    = PROJECT / "figures"

sys.path.insert(0, "/home/hmacgregor/BERIL-research-observatory/tools")
sys.path.insert(0, str(PROJECT / "scripts"))
from figure_style import apply_style, save, PALETTE, FIGW, ROW_H, grid_h
apply_style()
from pgls_utils import run_pgls

TREE_BAC = DATA / "gtdb_bac_genus_pruned.tree"
assert TREE_BAC.exists(), f"Tree not found: {TREE_BAC}"

_SPARK_AVAILABLE = False; _spark = None
try:
    from berdl_utils import get_spark_session
    _spark = get_spark_session()
    _SPARK_AVAILABLE = True
    print("Spark: connected")
except BaseException as _e:
    print(f"Spark unavailable (Spark sections will be skipped): {_e}")


[berdl_utils] JupyterHub SparkSession acquired: 4.0.1
Spark: connected


In [2]:
p1 = pd.read_csv(DATA / "01_pgls_input_bacteria.csv")
p1["genus_lower"] = p1["genus_lower"].str.lower().str.strip()
print(f"P1 genera: {len(p1)}")

def merge_p1(df, on="genus_lower"):
    m = p1.merge(df, on=on, how="inner")
    return m.dropna(subset=["ko_per_mb_primary", "mean_levins_B_std"])

def z_score(s):
    v = s.dropna(); return (s - v.mean()) / v.std()

def pca1(df, cols):
    sub = df[cols].copy()
    mask = sub.notna().any(axis=1)
    sub  = sub[mask].fillna(sub.median())
    sc   = StandardScaler().fit_transform(sub)
    pca  = PCA(n_components=1)
    pc1  = pca.fit_transform(sc)[:, 0]
    out  = pd.Series(np.nan, index=df.index)
    out.loc[sub.index] = pc1
    return out, pca.explained_variance_ratio_[0]

def extract_pgls(res, label):
    if res is None:
        return dict(label=label, beta=np.nan, SE=np.nan, p=np.nan, n=np.nan, lam=np.nan)
    betas = res.get("betas", {}) or {}
    SEs   = res.get("SEs", {}) or {}
    ps    = res.get("p_values", {}) or {}
    key   = list(betas.keys())[0] if betas else "niche"
    return dict(
        label = label,
        beta  = betas.get(key, res.get("beta", np.nan)),
        SE    = SEs.get(key, res.get("SE", np.nan)),
        p     = ps.get(key, res.get("p_value", np.nan)),
        n     = res.get("n", np.nan),
        lam   = res.get("lambda_est", np.nan),
    )

def sig_label(p):
    if np.isnan(p): return "ns"
    return "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))

def levins_b_std(row):
    row = row[row > 0]
    if len(row) < 2:
        return np.nan
    p = row / row.sum()
    B = 1.0 / (p ** 2).sum()
    return (B - 1) / (len(row) - 1)

def parquet_levins_pgls(parquet_path, class_col, label, layer_id, data_type="Other"):
    import os
    if not os.path.exists(parquet_path):
        print(f"  {label}: parquet not found — skip")
        return None
    df_raw = pd.read_parquet(parquet_path)
    if class_col not in df_raw.columns:
        candidates = [c for c in df_raw.columns if c not in ["genus_lower", "n_samples"]]
        if candidates:
            df_raw = df_raw.rename(columns={candidates[0]: class_col})
    pivot = df_raw.pivot_table(
        index="genus_lower", columns=class_col,
        values="n_samples", aggfunc="sum", fill_value=0,
    )
    B = pivot.apply(levins_b_std, axis=1).rename("B_cat").dropna().reset_index()
    df_pg = merge_p1(B)
    df_pg["niche"] = z_score(df_pg["B_cat"])
    df_pg = df_pg.dropna(subset=["niche"])
    if len(df_pg) < 50:
        print(f"  {label}: n={len(df_pg)} < 50 — skip PGLS")
        return None
    res = run_pgls(df_pg, str(TREE_BAC), response="ko_per_mb_primary",
                   predictors=["niche"], taxon_col="genus_lower",
                   label=layer_id, min_n=30)
    r = extract_pgls(res, label)
    r.update({"ci_lo": r["beta"] - 1.96 * r["SE"],
               "ci_hi": r["beta"] + 1.96 * r["SE"],
               "sig": sig_label(r["p"]), "layer": layer_id, "data_type": data_type})
    print(f"  {label}: beta={r['beta']:+.4f}  p={r['p']:.3e}  n={r['n']}  {r['sig']}")
    return r


P1 genera: 1574


## Section A: KO Subcategory PGLS

Tests whether the niche breadth → fewer metal genes relationship differs across KO functional categories (Resistance/Detoxification, Transport/Homeostasis, Sensing/Regulation, Metal-dependent Metabolism, Cofactor Biosynthesis).

In [3]:
# ── A1: Load KO presence matrix + subcategory metadata ──────────────────────
nb25 = pd.read_parquet(DATA / "nb25_ko_presence_matrix.parquet")
nb25["genus_lower"] = nb25["genus_lower"].str.replace(r"^g__", "", regex=True).str.lower().str.strip()

s8 = pd.read_csv(DATA / "s8_ko_metadata.csv")
print(f"s8 Tier 1+2 KOs: {len(s8)}")
print(s8["primary_category"].value_counts().to_string())

gstats = pd.read_csv(DATA / "01_genus_ko_density_spark.csv")[
    ["genus_lower", "n_genomes", "mean_genome_mb"]
]

# Restrict to Tier 1+2 KOs; merge subcategory labels
nb25_t12 = nb25.merge(s8[["ko", "primary_category"]], on="ko", how="inner")
nb25_t12 = nb25_t12.merge(gstats, on="genus_lower", how="inner")
print(f"\nnb25 (Tier 1+2): {len(nb25_t12)} rows, {nb25_t12.genus_lower.nunique()} genera")

# Proportion-based density: fraction of genus genomes that have each KO
nb25_t12["prop"] = nb25_t12["n_genomes_with_ko"] / nb25_t12["n_genomes"]

# Sum proportions per genus × subcategory, then normalise by genome size
subcat_sum = (
    nb25_t12.groupby(["genus_lower", "primary_category", "mean_genome_mb"])["prop"]
    .sum()
    .reset_index()
)
subcat_sum["subcat_per_mb"] = subcat_sum["prop"] / subcat_sum["mean_genome_mb"]

subcat_wide = subcat_sum.pivot_table(
    index=["genus_lower", "mean_genome_mb"],
    columns="primary_category",
    values="subcat_per_mb",
    fill_value=0,
).reset_index()
subcat_wide.columns.name = None
# Sanitise column names
subcat_wide.columns = [
    c.replace("/", "_").replace(" ", "_").replace("-", "_") for c in subcat_wide.columns
]
SUBCAT_COLS = [c for c in subcat_wide.columns if c not in ["genus_lower", "mean_genome_mb"]]
print(f"\nSubcategory columns: {SUBCAT_COLS}")
print(f"Genera in wide table: {len(subcat_wide)}")


s8 Tier 1+2 KOs: 118
primary_category
Resistance/Detoxification     43
Transport/Homeostasis         42
Metal-dependent Metabolism    15
Sensing/Regulation            14
Cofactor Biosynthesis          4



nb25 (Tier 1+2): 170536 rows, 8195 genera

Subcategory columns: ['Cofactor_Biosynthesis', 'Metal_dependent_Metabolism', 'Resistance_Detoxification', 'Sensing_Regulation', 'Transport_Homeostasis']
Genera in wide table: 8195


In [4]:
# ── A2: PGLS — ko_per_mb per subcategory ~ mean_levins_B_std ────────────────
df_sub = p1.merge(subcat_wide, on="genus_lower", how="inner").dropna(
    subset=["mean_levins_B_std"]
)
df_sub["B_z"] = z_score(df_sub["mean_levins_B_std"])
print(f"Genera for subcategory PGLS: {len(df_sub)}")

subcat_results = []
for col in SUBCAT_COLS:
    sub = df_sub.dropna(subset=[col, "B_z"]).copy()
    sub["response_z"] = z_score(sub[col])
    sub = sub.dropna(subset=["response_z"])
    if len(sub) < 50:
        print(f"  {col}: only {len(sub)} genera — skip")
        continue
    res = run_pgls(
        sub, str(TREE_BAC), response="response_z",
        predictors=["B_z"], taxon_col="genus_lower",
        label=f"A_{col}", min_n=30,
    )
    r = extract_pgls(res, col)
    r["subcategory"] = col.replace("_", " ")
    r["n_genera"] = len(sub)
    r["ci_lo"] = r["beta"] - 1.96 * r["SE"]
    r["ci_hi"] = r["beta"] + 1.96 * r["SE"]
    r["sig"] = sig_label(r["p"])
    subcat_results.append(r)
    print(f"  {col}: beta={r['beta']:+.4f}  p={r['p']:.3e}  n={r['n']}  {r['sig']}")

subcat_df = pd.DataFrame(subcat_results).sort_values("beta")
subcat_df.to_csv(DATA / "38_ko_subcategory_pgls.csv", index=False)
print(f"\nSaved {len(subcat_df)} results → 38_ko_subcategory_pgls.csv")


Genera for subcategory PGLS: 1573


  Cofactor_Biosynthesis: beta=-0.0404  p=1.056e-01  n=1573  ns


  Metal_dependent_Metabolism: beta=+0.0065  p=7.977e-01  n=1573  ns


  Resistance_Detoxification: beta=+0.0669  p=1.290e-02  n=1573  *


  Sensing_Regulation: beta=+0.0713  p=7.201e-03  n=1573  **


  Transport_Homeostasis: beta=-0.0004  p=9.883e-01  n=1573  ns

Saved 5 results → 38_ko_subcategory_pgls.csv


In [5]:
# ── A3: Figure 1 — KO subcategory forest plot ───────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(FIGW["1.5col"], ROW_H))
grid_h(ax)

y = list(range(len(subcat_df)))
colors = [PALETTE[0] if r["sig"] != "ns" else "#aaaaaa" for _, r in subcat_df.iterrows()]
xerr_lo = subcat_df["beta"] - subcat_df["ci_lo"]
xerr_hi = subcat_df["ci_hi"] - subcat_df["beta"]

ax.barh(
    y, subcat_df["beta"],
    xerr=[xerr_lo, xerr_hi],
    color=colors, edgecolor="k", linewidth=0.5,
    capsize=3, height=0.6,
    error_kw={"elinewidth": 0.8, "ecolor": "#444"},
)
ax.axvline(0, color="gray", lw=0.8, ls="--")
ax.set_yticks(y)
ax.set_yticklabels(subcat_df["subcategory"].tolist(), fontsize=8)
ax.set_xlabel("PGLS β (subcategory KO density per Mb ~ habitat niche breadth)", fontsize=9)
ax.set_ylabel("")

for i, (_, row) in enumerate(subcat_df.iterrows()):
    if row["sig"] != "ns":
        ax.annotate(row["sig"], xy=(row["ci_hi"] + 0.003, i),
                    fontsize=7, va="center", color="#333")
    ax.annotate(f"n={int(row['n_genera'])}",
                xy=(ax.get_xlim()[0] + 0.002, i - 0.35),
                fontsize=7, color="#808080")

ax.set_title("Metal gene subcategory ~ habitat niche breadth (PGLS)", fontsize=10)
fig.suptitle("A: KO Subcategory PGLS", y=1.02, fontsize=11, fontweight="bold")
save(fig, FIGS / "fig_nb38_subcategory_forest")
print("Saved fig_nb38_subcategory_forest.pdf")


Saved fig_nb38_subcategory_forest.pdf


## Section B: Categorical Levins B — Off-cluster Sources

Computes Levins B_std from 11 within-soil habitat category types (field, forest, tundra, paddy, etc.).

In [6]:
# ── B1: Soil habitat categorical Levins B (11 env_cat types) ────────────────
soil_env = pd.read_csv(DATA / "soil_sample_genus_env_counts.csv")
print(f"soil_env: {soil_env.shape}  env_cats: {sorted(soil_env.env_cat.unique())}")

pivot = soil_env.pivot_table(
    index="genus_lower", columns="env_cat",
    values="n_samples", aggfunc="sum", fill_value=0,
)

B_soil = pivot.apply(levins_b_std, axis=1).rename("B_soil_habitat").dropna()
B_soil_df = B_soil.reset_index()
B_soil_df.to_csv(DATA / "38_genus_soil_habitat_levins_b.csv", index=False)
print(f"Levins B_std: {len(B_soil_df)} genera  "
      f"range=[{B_soil_df.B_soil_habitat.min():.3f}, {B_soil_df.B_soil_habitat.max():.3f}]")

df_b1 = merge_p1(B_soil_df)
df_b1["niche"] = z_score(df_b1["B_soil_habitat"])
df_b1 = df_b1.dropna(subset=["niche"])
print(f"PGLS N: {len(df_b1)}")

res_b1 = run_pgls(
    df_b1, str(TREE_BAC), response="ko_per_mb_primary",
    predictors=["niche"], taxon_col="genus_lower",
    label="B1_soil_habitat_levins", min_n=50,
)
r_b1 = extract_pgls(res_b1, "B1: Soil habitat cat. Levins B")
r_b1.update({
    "ci_lo": r_b1["beta"] - 1.96 * r_b1["SE"],
    "ci_hi": r_b1["beta"] + 1.96 * r_b1["SE"],
    "sig": sig_label(r_b1["p"]),
    "layer": "B1_soil_habitat",
    "data_type": "Habitat",
})
print(f"B1 result: beta={r_b1['beta']:+.5f}  p={r_b1['p']:.3e}  n={r_b1['n']}  {r_b1['sig']}")

# Initialise comparison results collector
_comp_results = [r_b1]


soil_env: (19682, 4)  env_cats: ['aquatic', 'farm', 'field', 'flower', 'forest', 'general', 'leaf', 'paddy', 'plant', 'soil', 'tundra']


Levins B_std: 2981 genera  range=[0.004, 1.000]
PGLS N: 1526


B1 result: beta=-0.30882  p=5.191e-05  n=1526  ***


## Section C: Categorical Levins B — Spark Sources

All C0–C5 cells require the JupyterHub cluster. Results are saved as parquets; C6 computes Levins B and runs PGLS off-cluster after downloading the parquets.

In [7]:
# ── C0: Spark session status ─────────────────────────────────────────────────
if _SPARK_AVAILABLE:
    try:
        import pyspark.sql.functions as F
        from pyspark.sql.types import StringType
        _tables = [r.tableName for r in _spark.sql("SHOW TABLES IN arkinlab.envdbs").collect()]
        _needed = {"global_landcover_esa_2_0deg", "worldclim_master",
                   "global_population_density", "gemas", "soilgrids_master"}
        print("Required arkinlab tables found:", [t for t in _tables if t in _needed])
        print("Missing:", _needed - set(_tables))
        # Test microbeatlas access (required by C1-C5, D2)
        try:
            _spark.sql("SELECT 1 FROM arkinlab.microbeatlas.otu_counts_long LIMIT 1").collect()
            print("arkinlab.microbeatlas tables: accessible")
            print("Spark available — running Sections C, D")
        except Exception as _me:
            _SPARK_AVAILABLE = False
            print(f"arkinlab.microbeatlas.otu_counts_long not accessible: {type(_me).__name__}")
            print("Disabling Spark — Sections C and D will be skipped.")
    except Exception as _ce:
        _SPARK_AVAILABLE = False
        print(f"Spark tables not accessible: {type(_ce).__name__}: {str(_ce)[:200]}")
        print("Disabling Spark — Sections C and D will be skipped.")
else:
    print("Spark unavailable — Sections C and D will be skipped.")


Required arkinlab tables found: ['gemas', 'global_landcover_esa_2_0deg', 'global_population_density', 'soilgrids_master', 'worldclim_master']
Missing: set()


arkinlab.microbeatlas tables: accessible
Spark available — running Sections C, D


In [8]:
# ── C1: ESA CCI Land Cover categorical Levins B (Spark) ────────────────────
# All-biome + biome-stratified variants (soil/marine/freshwater)
# ESA CCI grid: lat at ODD degrees (-89,-87,...,87,89); lon at EVEN degrees (-180,-178,...,178,180)
# Snap formula: lat → ROUND((lat+1)/2)*2-1; lon → ROUND(lon/2)*2
if not _SPARK_AVAILABLE:
    print("Spark unavailable — skipping C1 (ESA CCI land cover)")
else:
    # MicrobeAtlas: snap to ESA CCI grid (lat=odd, lon=even)
    otu_meta = _spark.sql('''
        SELECT
            LOWER(TRIM(get(SPLIT(om.Tax, ';'), 5)))                        AS genus_lower,
            o.count,
            ROUND((sm.LatFieldValue + 1.0) / 2.0) * 2.0 - 1.0          AS lat_esa,
            ROUND(sm.LonFieldValue / 2.0) * 2.0                          AS lon_esa,
            LOWER(TRIM(sm.Env_Level_1))                                  AS biome
        FROM arkinlab.microbeatlas.otu_counts_long o
        JOIN arkinlab.microbeatlas.otu_metadata om ON o.otu_id = om.otu_id
        JOIN arkinlab.microbeatlas.sample_metadata sm ON o.sample_id = sm.sample_id
        WHERE o.count > 0
          AND sm.LatFieldValue IS NOT NULL
          AND sm.LonFieldValue IS NOT NULL
          AND get(SPLIT(om.Tax, ';'), 5) IS NOT NULL
          AND LENGTH(TRIM(get(SPLIT(om.Tax, ';'), 5))) > 0
    ''')

    # ESA CCI: cast string lat/lon to double (already at grid resolution)
    esa = (
        _spark.table("arkinlab.envdbs.global_landcover_esa_2_0deg")
        .filter(F.col("landcover_class").isNotNull())
        .withColumn("lat_esa", F.col("lat").cast("double"))
        .withColumn("lon_esa", F.col("lon").cast("double"))
        .drop("lat", "lon")
    )

    joined_esa = otu_meta.join(esa, on=["lat_esa", "lon_esa"], how="inner")

    # All-biome
    genus_lc = (
        joined_esa
        .groupBy("genus_lower", "landcover_class")
        .agg(F.count("*").alias("n_samples"))
    )
    pd_lc = genus_lc.toPandas(); pd_lc.attrs = {}
    pd_lc.to_parquet(DATA / "38_genus_landcover_counts.parquet", index=False)
    print(f"ESA CCI all-biome: {pd_lc.genus_lower.nunique()} genera")

    # Biome-stratified
    # Note: MicrobeAtlas has no marine/ocean Env_Level_1 — marine count will be 0
    for biome_key, biome_label in [("soil", "soil"), ("marine", "marine"), ("aquatic", "freshwater")]:
        pd_b = (
            joined_esa.filter(F.col("biome").contains(biome_key))
            .groupBy("genus_lower", "landcover_class")
            .agg(F.count("*").alias("n_samples"))
            .toPandas()
        )
        pd_b.attrs = {}
        pd_b.to_parquet(DATA / f"38_genus_landcover_{biome_label}_counts.parquet", index=False)
        print(f"  {biome_label}: {pd_b.genus_lower.nunique()} genera")

    print("C1 complete.")


ESA CCI all-biome: 0 genera


  soil: 0 genera


  marine: 0 genera


  freshwater: 0 genera
C1 complete.


In [9]:
# ── C2: Köppen-Geiger categorical Levins B (derived from WorldClim, Spark) ──
if not _SPARK_AVAILABLE:
    print("Spark unavailable — skipping C2 (Köppen-Geiger)")
else:
    from pyspark.sql.functions import udf
    from pyspark.sql.types import StringType

    @udf(StringType())
    def kg_class_udf(bio1, bio12):
        # bio1 = MAT × 10 (°C×10), bio12 = MAP (mm)
        if bio1 is None or bio12 is None:
            return None
        mat = float(bio1) / 10.0
        prec = float(bio12)
        if prec < 200:      return "Arid"
        elif mat >= 18:     return "Tropical"
        elif mat >= 10:     return "Temperate"
        elif mat >= 0:      return "Continental"
        else:               return "Polar"

    wc = (
        _spark.table("arkinlab.envdbs.worldclim_master")
        .select("lat", "lon", "bio_1", "bio_12")
        .withColumn("kg_class", kg_class_udf(F.col("bio_1"), F.col("bio_12")))
        .filter(F.col("kg_class").isNotNull())
        .withColumnRenamed("lat", "lat_025")
        .withColumnRenamed("lon", "lon_025")
    )

    otu_meta_wc = _spark.sql('''
        SELECT
            LOWER(TRIM(get(SPLIT(om.Tax, ';'), 5)))      AS genus_lower,
            o.count,
            ROUND(sm.LatFieldValue / 0.25) * 0.25  AS lat_025,
            ROUND(sm.LonFieldValue / 0.25) * 0.25  AS lon_025
        FROM arkinlab.microbeatlas.otu_counts_long o
        JOIN arkinlab.microbeatlas.otu_metadata om ON o.otu_id = om.otu_id
        JOIN arkinlab.microbeatlas.sample_metadata sm ON o.sample_id = sm.sample_id
        WHERE o.count > 0
          AND sm.LatFieldValue IS NOT NULL
          AND sm.LonFieldValue IS NOT NULL
          AND get(SPLIT(om.Tax, ';'), 5) IS NOT NULL
          AND LENGTH(TRIM(get(SPLIT(om.Tax, ';'), 5))) > 0
    ''')

    joined_kg = otu_meta_wc.join(wc, on=["lat_025", "lon_025"], how="inner")
    genus_kg = (
        joined_kg
        .groupBy("genus_lower", "kg_class")
        .agg(F.count("*").alias("n_samples"))
    )
    kg_pd = genus_kg.toPandas(); kg_pd.attrs = {}
    kg_pd.to_parquet(DATA / "38_genus_kg_counts.parquet", index=False)
    print(f"KG classes: {sorted(kg_pd.kg_class.unique())}")
    print(f"KG: {kg_pd.genus_lower.nunique()} genera")
    print("C2 complete.")


KG classes: ['Arid', 'Continental', 'Polar']
KG: 3430 genera
C2 complete.


In [10]:
# ── C3: Population density quintile categorical Levins B (Spark) ─────────────
if not _SPARK_AVAILABLE:
    print("Spark unavailable — skipping C3 (population density)")
else:
    from pyspark.sql.functions import udf
    from pyspark.sql.types import StringType

    @udf(StringType())
    def pop_class_udf(pop):
        if pop is None:   return None
        pop = float(pop)
        if pop == 0:      return "1_wilderness"
        elif pop < 10:    return "2_rural"
        elif pop < 100:   return "3_suburban"
        elif pop < 1000:  return "4_urban"
        else:             return "5_megacity"

    pop = (
        _spark.table("arkinlab.envdbs.global_population_density")
        .withColumn("pop_class", pop_class_udf(F.col("population_density")))
        .filter(F.col("pop_class").isNotNull())
        .select(
            (F.round(F.col("lat") / 0.25) * 0.25).alias("lat_025"),
            (F.round(F.col("lon") / 0.25) * 0.25).alias("lon_025"),
            "pop_class",
        )
    )

    otu_meta_pop = _spark.sql('''
        SELECT
            LOWER(TRIM(get(SPLIT(om.Tax, ';'), 5)))      AS genus_lower,
            o.count,
            ROUND(sm.LatFieldValue / 0.25) * 0.25  AS lat_025,
            ROUND(sm.LonFieldValue / 0.25) * 0.25  AS lon_025
        FROM arkinlab.microbeatlas.otu_counts_long o
        JOIN arkinlab.microbeatlas.otu_metadata om ON o.otu_id = om.otu_id
        JOIN arkinlab.microbeatlas.sample_metadata sm ON o.sample_id = sm.sample_id
        WHERE o.count > 0
          AND sm.LatFieldValue IS NOT NULL
          AND sm.LonFieldValue IS NOT NULL
          AND get(SPLIT(om.Tax, ';'), 5) IS NOT NULL
          AND LENGTH(TRIM(get(SPLIT(om.Tax, ';'), 5))) > 0
    ''')

    joined_pop = otu_meta_pop.join(pop, on=["lat_025", "lon_025"], how="inner")
    genus_pop = (
        joined_pop
        .groupBy("genus_lower", "pop_class")
        .agg(F.count("*").alias("n_samples"))
    )
    pop_pd = genus_pop.toPandas(); pop_pd.attrs = {}
    pop_pd.to_parquet(DATA / "38_genus_popdens_counts.parquet", index=False)
    print(f"Pop density: {pop_pd.genus_lower.nunique()} genera")
    print("C3 complete.")


Pop density: 3430 genera
C3 complete.


In [11]:
# ── C4: GEMAS + NGSA combined geochemical niche (Spark) ──────────────────────
# Pairs European GEMAS with USA NGSA; also adds Fe, Mn to NGSA metal set
if not _SPARK_AVAILABLE:
    print("Spark unavailable — skipping C4 (GEMAS + NGSA combined)")
else:
    # Discover GEMAS schema
    gemas_schema = _spark.sql("DESCRIBE arkinlab.envdbs.gemas").toPandas()
    all_gemas_cols = gemas_schema["col_name"].tolist()
    print("GEMAS all columns:", all_gemas_cols[:30])

    METALS = ["Cu", "Ni", "Zn", "Pb", "As", "Co", "Cr", "Hg", "Fe", "Mn"]
    GEMAS_METAL_COLS = [c for c in all_gemas_cols
                        if any(m == c[:len(m)] for m in METALS)
                        and c not in ["X_Coo", "Y_Coo"]]
    print(f"GEMAS metal cols identified: {GEMAS_METAL_COLS}")

    if not GEMAS_METAL_COLS:
        print("WARNING: No metal columns identified in GEMAS — check schema above and adjust METALS list")
    else:
        # Snap GEMAS to 1° grid
        gemas = _spark.table("arkinlab.envdbs.gemas").select(
            (F.round(F.col("Y_Coo")).cast("double")).alias("lat_1deg"),
            (F.round(F.col("X_Coo")).cast("double")).alias("lon_1deg"),
            *[F.col(c).cast("double") for c in GEMAS_METAL_COLS],
        ).filter(F.col("lat_1deg").isNotNull())

        # MicrobeAtlas soil samples → 1° grid (3-way join for genus + lat/lon/biome)
        otu_soil = _spark.sql('''
            SELECT
                LOWER(TRIM(get(SPLIT(om.Tax, ';'), 5))) AS genus_lower,
                o.count,
                ROUND(sm.LatFieldValue)            AS lat_1deg,
                ROUND(sm.LonFieldValue)            AS lon_1deg
            FROM arkinlab.microbeatlas.otu_counts_long o
            JOIN arkinlab.microbeatlas.otu_metadata om ON o.otu_id = om.otu_id
            JOIN arkinlab.microbeatlas.sample_metadata sm ON o.sample_id = sm.sample_id
            WHERE o.count > 0
              AND sm.LatFieldValue IS NOT NULL
              AND sm.LonFieldValue IS NOT NULL
              AND get(SPLIT(om.Tax, ';'), 5) IS NOT NULL
              AND LENGTH(TRIM(get(SPLIT(om.Tax, ';'), 5))) > 0
              AND LOWER(sm.Env_Level_1) LIKE '%soil%'
        ''')

        # Per-grid-cell GEMAS metal averages
        gemas_grid = gemas.groupBy("lat_1deg", "lon_1deg").agg(
            *[F.avg(c).alias(f"gemas_{c}") for c in GEMAS_METAL_COLS]
        )

        joined_gemas = otu_soil.join(gemas_grid, on=["lat_1deg", "lon_1deg"], how="inner")

        # Per-genus SD of each metal across occupied GEMAS grid cells
        gemas_metal_renamed = [f"gemas_{c}" for c in GEMAS_METAL_COLS]
        genus_gemas = joined_gemas.groupBy("genus_lower").agg(
            F.count("*").alias("n_cells"),
            *[F.stddev(c).alias(c + "_sd") for c in gemas_metal_renamed],
        ).filter(F.col("n_cells") >= 5)

        gemas_pd = genus_gemas.toPandas(); gemas_pd.attrs = {}
        gemas_pd.to_csv(DATA / "38_gemas_genus_metal_sd.csv", index=False)
        print(f"GEMAS: {len(gemas_pd)} genera saved to 38_gemas_genus_metal_sd.csv")

    # ── NGSA: check if Fe, Mn ICP-MS columns available in Spark ─────────────
    try:
        ngsa_schema = _spark.sql("DESCRIBE arkinlab.envdbs.ngsa_genus_sd").toPandas()
        ngsa_fe_mn = [c for c in ngsa_schema["col_name"]
                      if any(m in c for m in ["Fe_ICP", "Mn_ICP"]) and "_sd" in c]
        if ngsa_fe_mn:
            cols_str = ", ".join(ngsa_fe_mn)
            ngsa_ext = _spark.sql(f"SELECT genus_lower, {cols_str} FROM arkinlab.envdbs.ngsa_genus_sd").toPandas()
            ngsa_ext.attrs = {}
            ngsa_base = pd.read_csv(DATA / "env_niche_ngsa_spark.csv")
            ngsa_merged = ngsa_base.merge(ngsa_ext, on="genus_lower", how="left")
            ngsa_merged.to_csv(DATA / "38_ngsa_extended_genus_metal_sd.csv", index=False)
            print(f"NGSA extended (+Fe/Mn): {len(ngsa_merged)} genera")
        else:
            print("Fe/Mn not found in ngsa_genus_sd — copying base NGSA")
            import shutil
            shutil.copy(DATA / "env_niche_ngsa_spark.csv",
                        DATA / "38_ngsa_extended_genus_metal_sd.csv")
    except Exception as e:
        print(f"NGSA Spark extension skipped: {e}")
        import shutil
        shutil.copy(DATA / "env_niche_ngsa_spark.csv",
                    DATA / "38_ngsa_extended_genus_metal_sd.csv")

    print("C4 complete.")


GEMAS all columns: ['id', 'country', 'country_id', 'type', 'type2', 'longitude', 'latitude', 'x_laea', 'y_laea', 'altitude', 'aps_1960_1990', 'amt_1960_1990', 'aps_1970_2000', 'amt_1970_2000', 'climate', 'soiltype', 'soilclass', 'cgsg', 'litho', 'pm_hart', 'pm_guen', 'ecoregio', 'pd_2005', 'pd_2020', 'ag_ppm_ar', 'al_ppm_ar', 'as_ppm_ar', 'au_ppm_ar', 'b_ppm_ar', 'ba_ppm_ar']
GEMAS metal cols identified: []


{"ts": "2026-08-02 01:48:18.795", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[TABLE_OR_VIEW_NOT_FOUND] The table or view `arkinlab`.`envdbs`.`ngsa_genus_sd` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 1 pos 9;\n'DescribeRelation false, [col_name#78943, data_type#78944, comment#78945]\n+- 'UnresolvedTableOrView [arkinlab, envdbs, ngsa_genus_sd], DESCRIBE TABLE, true\n\n\nJVM stacktrace:\norg.apache.spark.sql.catalyst.ExtendedAnalysisException\n\tat org.apache.spark.sql.catalyst.analysis.package$AnalysisErrorAt.tableNotFound(package.scala:91)\n\tat org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis0$2(CheckAnalysis.scala:303)\n\tat org.apache.spark.sql.catalyst.analysis.C

NGSA Spark extension skipped: [TABLE_OR_VIEW_NOT_FOUND] The table or view `arkinlab`.`envdbs`.`ngsa_genus_sd` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 1 pos 9;
'DescribeRelation false, [col_name#78943, data_type#78944, comment#78945]
+- 'UnresolvedTableOrView [arkinlab, envdbs, ngsa_genus_sd], DESCRIBE TABLE, true


JVM stacktrace:
org.apache.spark.sql.catalyst.ExtendedAnalysisException
	at org.apache.spark.sql.catalyst.analysis.package$AnalysisErrorAt.tableNotFound(package.scala:91)
	at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis0$2(CheckAnalysis.scala:303)
	at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis0$2$adapted(CheckAnalysis.scala:284)
	at org

In [12]:
# ── C5: SoilGrids CEC + Clay categorical Levins B (Spark) ───────────────────
if not _SPARK_AVAILABLE:
    print("Spark unavailable — skipping C5 (SoilGrids CEC + clay)")
else:
    sg_schema = _spark.sql("DESCRIBE arkinlab.envdbs.soilgrids_master").toPandas()
    sg_cols = sg_schema["col_name"].tolist()
    print("SoilGrids columns (first 25):", sg_cols[:25])

    cec_col  = next((c for c in sg_cols if "cation_exchange" in c.lower() and "_0cm" in c), None)
    clay_col = next((c for c in sg_cols if c.startswith("clay") and "_0cm" in c), None)
    p_col    = next((c for c in sg_cols if "phosph" in c.lower()), None)
    print(f"Using: cec={cec_col}, clay={clay_col}, phosphorus={p_col}")

    if cec_col is None or clay_col is None:
        print("CEC or clay column not found — skip C5. Check column names above.")
    else:
        from pyspark.sql.functions import udf
        from pyspark.sql.types import StringType

        @udf(StringType())
        def cec_class_udf(cec):
            # SoilGrids CEC in cmol(c)/kg; data range 0-6.4; quintile-based thresholds
            if cec is None: return None
            v = float(cec)
            if v < 0.1:   return "1_very_low"
            elif v < 0.3: return "2_low"
            elif v < 0.7: return "3_medium"
            elif v < 1.1: return "4_high"
            else:         return "5_very_high"

        @udf(StringType())
        def clay_class_udf(clay):
            # SoilGrids clay_0cm is in % (0-75); no unit conversion needed
            if clay is None: return None
            v = float(clay)
            if v < 10:   return "1_sand"
            elif v < 25: return "2_loam"
            elif v < 40: return "3_clay_loam"
            else:        return "4_clay"

        sg = (
            _spark.table("arkinlab.envdbs.soilgrids_master")
            .select(
                (F.round(F.col("lat") / 0.25) * 0.25).alias("lat_025"),
                (F.round(F.col("lon") / 0.25) * 0.25).alias("lon_025"),
                cec_class_udf(F.col(cec_col)).alias("cec_class"),
                clay_class_udf(F.col(clay_col)).alias("clay_class"),
            )
            .filter(F.col("cec_class").isNotNull())
        )

        otu_meta_sg = _spark.sql('''
            SELECT
                LOWER(TRIM(get(SPLIT(om.Tax, ';'), 5)))      AS genus_lower,
                o.count,
                ROUND(sm.LatFieldValue / 0.25) * 0.25  AS lat_025,
                ROUND(sm.LonFieldValue / 0.25) * 0.25  AS lon_025
            FROM arkinlab.microbeatlas.otu_counts_long o
            JOIN arkinlab.microbeatlas.otu_metadata om ON o.otu_id = om.otu_id
            JOIN arkinlab.microbeatlas.sample_metadata sm ON o.sample_id = sm.sample_id
            WHERE o.count > 0
              AND sm.LatFieldValue IS NOT NULL
              AND sm.LonFieldValue IS NOT NULL
              AND get(SPLIT(om.Tax, ';'), 5) IS NOT NULL
              AND LENGTH(TRIM(get(SPLIT(om.Tax, ';'), 5))) > 0
        ''')

        joined_sg = otu_meta_sg.join(sg, on=["lat_025", "lon_025"], how="inner")

        for class_col, suffix in [("cec_class", "cec"), ("clay_class", "clay")]:
            gc = (
                joined_sg
                .groupBy("genus_lower", class_col)
                .agg(F.count("*").alias("n_samples"))
                .withColumnRenamed(class_col, "env_class")
            )
            pd_gc = gc.toPandas(); pd_gc.attrs = {}
            pd_gc.to_parquet(DATA / f"38_genus_{suffix}_counts.parquet", index=False)
            print(f"SoilGrids {suffix}: {pd_gc.genus_lower.nunique()} genera")

    print("C5 complete.")


SoilGrids columns (first 25): ['lat', 'lon', 'pH_0-5cm', 'pH_5-15cm', 'pH_15-30cm', 'pH_30-60cm', 'pH_60-100cm', 'pH_100-200cm', 'soil_organic_carbon_0-5cm', 'soil_organic_carbon_5-15cm', 'soil_organic_carbon_15-30cm', 'soil_organic_carbon_30-60cm', 'soil_organic_carbon_60-100cm', 'soil_organic_carbon_100-200cm', 'bulk_density_0-5cm', 'bulk_density_5-15cm', 'bulk_density_15-30cm', 'bulk_density_30-60cm', 'bulk_density_60-100cm', 'bulk_density_100-200cm', 'pH_30cm', 'pH_60cm', 'pH_200cm', 'pH_10cm', 'pH_100cm']
Using: cec=cation_exchange_capacity_0cm, clay=clay_0cm, phosphorus=None


SoilGrids cec: 3430 genera


SoilGrids clay: 3430 genera
C5 complete.


### C6: Off-cluster Levins B + PGLS from Spark parquets

Download parquets from cluster to `data/` first, then run this cell.

In [13]:
# ── C6: Compute Levins B + PGLS from all Spark-saved parquets ───────────────
# Run this cell after downloading parquets from the cluster to DATA/.

SPARK_LAYERS = [
    # (parquet_path, class_col, label, layer_id, data_type)
    ("38_genus_landcover_counts.parquet",          "landcover_class", "C1a: ESA CCI land cover (all biomes)",  "C1a_ESA",      "Habitat"),
    ("38_genus_landcover_soil_counts.parquet",     "landcover_class", "C1b: ESA CCI (soil biome)",             "C1b_ESA_soil", "Habitat"),
    ("38_genus_landcover_marine_counts.parquet",   "landcover_class", "C1c: ESA CCI (marine biome)",           "C1c_ESA_mar",  "Habitat"),
    ("38_genus_landcover_freshwater_counts.parquet","landcover_class","C1d: ESA CCI (freshwater biome)",       "C1d_ESA_fw",   "Habitat"),
    ("38_genus_kg_counts.parquet",                 "kg_class",        "C2: Köppen-Geiger climate",             "C2_KG",        "Climate"),
    ("38_genus_popdens_counts.parquet",            "pop_class",       "C3: Population density",                "C3_pop",       "Anthropogenic"),
    ("38_genus_cec_counts.parquet",                "env_class",       "C5a: SoilGrids CEC",                   "C5a_CEC",      "Edaphic"),
    ("38_genus_clay_counts.parquet",               "env_class",       "C5b: SoilGrids clay",                  "C5b_clay",     "Edaphic"),
]

spark_results = []
for fname, cls, lab, lid, dt in SPARK_LAYERS:
    r = parquet_levins_pgls(str(DATA / fname), cls, lab, lid, dt)
    if r is not None:
        spark_results.append(r)

_comp_results.extend(spark_results)
print(f"\nSpark-derived results added: {len(spark_results)}")
print(f"Total _comp_results: {len(_comp_results)}")


  C1a: ESA CCI land cover (all biomes): n=0 < 50 — skip PGLS
  C1b: ESA CCI (soil biome): n=0 < 50 — skip PGLS
  C1c: ESA CCI (marine biome): n=0 < 50 — skip PGLS
  C1d: ESA CCI (freshwater biome): n=0 < 50 — skip PGLS


  C2: Köppen-Geiger climate: beta=+0.1902  p=2.701e-02  n=1573  *


  C3: Population density: beta=-0.0326  p=6.834e-01  n=1572  ns


  C5a: SoilGrids CEC: beta=+0.5901  p=2.014e-13  n=1573  ***


  C5b: SoilGrids clay: beta=+0.0757  p=3.312e-01  n=1572  ns

Spark-derived results added: 4
Total _comp_results: 5


### C4b: GEMAS + NGSA geochemical niche PGLS

Run after C4 completes on cluster and CSVs are downloaded.

In [14]:
# ── C4b: GEMAS + NGSA geochemical niche PGLS (off-cluster, post-cluster) ────
import os

# GEMAS-only (European soil geochemical niche)
if os.path.exists(DATA / "38_gemas_genus_metal_sd.csv"):
    gemas_pd = pd.read_csv(DATA / "38_gemas_genus_metal_sd.csv")
    gemas_sd_cols = [c for c in gemas_pd.columns if c.endswith("_sd")]
    print(f"GEMAS: {len(gemas_pd)} genera, {len(gemas_sd_cols)} metal SD cols: {gemas_sd_cols}")

    df_g = merge_p1(gemas_pd).dropna(subset=gemas_sd_cols[:3], how="all")
    valid_g = [c for c in gemas_sd_cols if df_g[c].notna().sum() > 50]
    if len(valid_g) >= 2:
        df_g["niche_pc1"], var_g = pca1(df_g, valid_g)
        df_g["niche"] = z_score(df_g["niche_pc1"])
        df_g = df_g.dropna(subset=["niche"])
        print(f"GEMAS PGLS: n={len(df_g)}, PC1 var={var_g:.1%} from {valid_g}")
        res_g = run_pgls(df_g, str(TREE_BAC), response="ko_per_mb_primary",
                         predictors=["niche"], taxon_col="genus_lower",
                         label="C4a_GEMAS_geochem", min_n=30)
        r_g = extract_pgls(res_g, "C4a: GEMAS geochem niche (Europe)")
        r_g.update({"ci_lo": r_g["beta"] - 1.96*r_g["SE"],
                    "ci_hi": r_g["beta"] + 1.96*r_g["SE"],
                    "sig": sig_label(r_g["p"]), "layer": "C4a_GEMAS", "data_type": "Geochemical"})
        print(f"GEMAS: beta={r_g['beta']:+.4f}  p={r_g['p']:.3e}  n={r_g['n']}  {r_g['sig']}")
        _comp_results.append(r_g)
    else:
        print("Insufficient valid metal columns for GEMAS PGLS")
else:
    print("38_gemas_genus_metal_sd.csv not found — run C4 on cluster first")

# NGSA extended (+ Fe, Mn if available)
ngsa_ext_path = DATA / "38_ngsa_extended_genus_metal_sd.csv"
if os.path.exists(ngsa_ext_path):
    ngsa_ext = pd.read_csv(ngsa_ext_path)
    ngsa_sd_cols = [c for c in ngsa_ext.columns if c.endswith("_sd") and "MMI" not in c]
    print(f"\nNGSA extended: {len(ngsa_ext)} genera, metals: {ngsa_sd_cols}")
    df_n = merge_p1(ngsa_ext).dropna(subset=ngsa_sd_cols[:3], how="all")
    valid_n = [c for c in ngsa_sd_cols if df_n[c].notna().sum() > 100]
    if len(valid_n) >= 2:
        df_n["niche_pc1"], var_n = pca1(df_n, valid_n)
        df_n["niche"] = z_score(df_n["niche_pc1"])
        df_n = df_n.dropna(subset=["niche"])
        print(f"NGSA ext PGLS: n={len(df_n)}, PC1 var={var_n:.1%}")
        res_n = run_pgls(df_n, str(TREE_BAC), response="ko_per_mb_primary",
                         predictors=["niche"], taxon_col="genus_lower",
                         label="C4b_NGSA_ext", min_n=30)
        r_n = extract_pgls(res_n, "C4b: NGSA geochem niche (+Fe/Mn, USA)")
        r_n.update({"ci_lo": r_n["beta"] - 1.96*r_n["SE"],
                    "ci_hi": r_n["beta"] + 1.96*r_n["SE"],
                    "sig": sig_label(r_n["p"]), "layer": "C4b_NGSA_ext", "data_type": "Geochemical"})
        print(f"NGSA ext: beta={r_n['beta']:+.4f}  p={r_n['p']:.3e}  n={r_n['n']}  {r_n['sig']}")
        _comp_results.append(r_n)
else:
    print("38_ngsa_extended_genus_metal_sd.csv not found — run C4 on cluster first")


38_gemas_genus_metal_sd.csv not found — run C4 on cluster first

NGSA extended: 3227 genera, metals: ['Cu_ICP_MS_mg_kg_0_2_sd', 'Ni_ICP_MS_mg_kg_0_5_sd', 'Zn_ICP_MS_mg_kg_0_9_sd', 'Pb_ICP_MS_mg_kg_0_1_sd', 'As_ICP_MS_mg_kg_0_4_sd', 'Co_ICP_MS_mg_kg_0_1_sd', 'Cr_ICP_MS_mg_kg_0_5_sd', 'Hg_AR_mg_kg_0_01_sd']
NGSA ext PGLS: n=1530, PC1 var=48.0%


NGSA ext: beta=+0.0376  p=6.408e-01  n=1530  ns


## Section D: Biome-stratified PGLS (Spark)

Restricts ke_pangenome genomes by GTDB isolation source (soil/marine/freshwater) for biome-specific metal gene densities. Pairs with biome-filtered MicrobeAtlas sample occurrence data.

In [15]:
# ── D1: Per-biome ko_per_mb from ke_pangenome (Spark) ────────────────────────
# NOTE: kbase.ke_pangenome.ko_annotations does not exist. The correct route is:
#   genome → gene (1B rows) → eggnog_mapper_annotations (KEGG_ko comma-separated)
# This 3-way join with EXPLODE is too expensive for the JupyterHub cluster timeout.
# D3 has a built-in fallback to ko_per_mb_primary when biome-specific CSVs are absent.
# Skip D1 silently — D3 will use the all-biome ko_per_mb_primary as the response.
if not _SPARK_AVAILABLE:
    print("Spark unavailable — skipping D1 (biome-stratified ko_per_mb)")
else:
    print("D1: biome-specific ko_per_mb skipped — gene table (1B rows) join infeasible.")
    print("  D3 will use ko_per_mb_primary fallback for all three biomes.")
    print("D1 complete (no-op).")


D1: biome-specific ko_per_mb skipped — gene table (1B rows) join infeasible.
  D3 will use ko_per_mb_primary fallback for all three biomes.
D1 complete (no-op).


In [16]:
# ── D2: Per-biome categorical Levins B from MicrobeAtlas (Spark) ─────────────
# Soil → land cover; Marine → KG climate; Freshwater → population density
# 3-way join: otu_counts_long × otu_metadata (genus via Tax) × sample_metadata (lat/lon/biome)
if not _SPARK_AVAILABLE:
    print("Spark unavailable — skipping D2 (biome-stratified Levins B)")
else:
    from pyspark.sql.functions import udf
    from pyspark.sql.types import StringType

    # ── Marine: KG class from WorldClim, restricted to marine samples ─────────
    @udf(StringType())
    def kg_udf(bio1, bio12):
        if bio1 is None or bio12 is None: return None
        mat = float(bio1) / 10.0; prec = float(bio12)
        if prec < 200:      return "Arid"
        elif mat >= 18:     return "Tropical"
        elif mat >= 10:     return "Temperate"
        elif mat >= 0:      return "Continental"
        else:               return "Polar"

    wc_kg = (
        _spark.table("arkinlab.envdbs.worldclim_master")
        .select("lat", "lon", "bio_1", "bio_12")
        .withColumn("kg_class", kg_udf(F.col("bio_1"), F.col("bio_12")))
        .filter(F.col("kg_class").isNotNull())
        .withColumnRenamed("lat", "lat_025")
        .withColumnRenamed("lon", "lon_025")
    )

    otu_marine = _spark.sql('''
        SELECT
            LOWER(TRIM(get(SPLIT(om.Tax, ';'), 5)))      AS genus_lower,
            o.count,
            ROUND(sm.LatFieldValue / 0.25) * 0.25  AS lat_025,
            ROUND(sm.LonFieldValue / 0.25) * 0.25  AS lon_025
        FROM arkinlab.microbeatlas.otu_counts_long o
        JOIN arkinlab.microbeatlas.otu_metadata om ON o.otu_id = om.otu_id
        JOIN arkinlab.microbeatlas.sample_metadata sm ON o.sample_id = sm.sample_id
        WHERE o.count > 0
          AND sm.LatFieldValue IS NOT NULL
          AND sm.LonFieldValue IS NOT NULL
          AND get(SPLIT(om.Tax, ';'), 5) IS NOT NULL
          AND LENGTH(TRIM(get(SPLIT(om.Tax, ';'), 5))) > 0
          AND LOWER(sm.Env_Level_1) RLIKE 'marine|ocean'
    ''')

    marine_kg = otu_marine.join(wc_kg, on=["lat_025", "lon_025"], how="inner")
    pd_mk = (marine_kg.groupBy("genus_lower", "kg_class")
                      .agg(F.count("*").alias("n_samples"))
                      .toPandas())
    pd_mk.attrs = {}
    pd_mk.to_parquet(DATA / "38_genus_kg_marine_counts.parquet", index=False)
    print(f"Marine KG: {pd_mk.genus_lower.nunique()} genera")

    # ── Freshwater: population density quintile, restricted to aquatic samples ─
    @udf(StringType())
    def pop_udf(pop):
        if pop is None:   return None
        pop = float(pop)
        if pop == 0:      return "1_wilderness"
        elif pop < 10:    return "2_rural"
        elif pop < 100:   return "3_suburban"
        elif pop < 1000:  return "4_urban"
        else:             return "5_megacity"

    pop2 = (
        _spark.table("arkinlab.envdbs.global_population_density")
        .withColumn("pop_class", pop_udf(F.col("population_density")))
        .filter(F.col("pop_class").isNotNull())
        .select(
            (F.round(F.col("lat") / 0.25) * 0.25).alias("lat_025"),
            (F.round(F.col("lon") / 0.25) * 0.25).alias("lon_025"),
            "pop_class",
        )
    )

    otu_fw = _spark.sql('''
        SELECT
            LOWER(TRIM(get(SPLIT(om.Tax, ';'), 5)))      AS genus_lower,
            o.count,
            ROUND(sm.LatFieldValue / 0.25) * 0.25  AS lat_025,
            ROUND(sm.LonFieldValue / 0.25) * 0.25  AS lon_025
        FROM arkinlab.microbeatlas.otu_counts_long o
        JOIN arkinlab.microbeatlas.otu_metadata om ON o.otu_id = om.otu_id
        JOIN arkinlab.microbeatlas.sample_metadata sm ON o.sample_id = sm.sample_id
        WHERE o.count > 0
          AND sm.LatFieldValue IS NOT NULL
          AND sm.LonFieldValue IS NOT NULL
          AND get(SPLIT(om.Tax, ';'), 5) IS NOT NULL
          AND LENGTH(TRIM(get(SPLIT(om.Tax, ';'), 5))) > 0
          AND LOWER(sm.Env_Level_1) RLIKE 'aquatic|freshwater|river|lake'
    ''')

    fw_pop = otu_fw.join(pop2, on=["lat_025", "lon_025"], how="inner")
    pd_fw = (fw_pop.groupBy("genus_lower", "pop_class")
                   .agg(F.count("*").alias("n_samples"))
                   .toPandas())
    pd_fw.attrs = {}
    pd_fw.to_parquet(DATA / "38_genus_popdens_freshwater_counts.parquet", index=False)
    print(f"Freshwater pop density: {pd_fw.genus_lower.nunique()} genera")

    print("D2 complete.")


Marine KG: 0 genera


Freshwater pop density: 3412 genera
D2 complete.


### D3: Biome-stratified PGLS

Run off-cluster after D1/D2 parquets are downloaded. Falls back to ko_per_mb_primary if biome-specific CSV is missing.

In [17]:
# ── D3: Biome-stratified PGLS ────────────────────────────────────────────────
import os

# If biome-specific ko_per_mb not available (D1 skipped), fall back to full ko_per_mb_primary
BIOME_CONFIGS = [
    # (biome, ko_csv,                              niche_parquet,                              class_col,        data_type)
    ("soil",       "38_genus_ko_per_mb_soil.csv",      "38_genus_landcover_soil_counts.parquet",   "landcover_class", "Habitat"),
    ("marine",     "38_genus_ko_per_mb_marine.csv",    "38_genus_kg_marine_counts.parquet",        "kg_class",        "Climate"),
    ("freshwater", "38_genus_ko_per_mb_freshwater.csv","38_genus_popdens_freshwater_counts.parquet","pop_class",       "Anthropogenic"),
]

biome_results = []
for biome, ko_csv, niche_parquet, class_col, dtype in BIOME_CONFIGS:
    niche_path = DATA / niche_parquet
    ko_path    = DATA / ko_csv
    fallback_ko = not os.path.exists(ko_path)

    if not os.path.exists(niche_path):
        print(f"  {biome}: niche parquet missing — skip")
        continue

    # Niche: categorical Levins B for this biome
    niche_raw = pd.read_parquet(niche_path)
    pivot_b = niche_raw.pivot_table(
        index="genus_lower", columns=class_col,
        values="n_samples", aggfunc="sum", fill_value=0,
    )
    B_b = pivot_b.apply(levins_b_std, axis=1).rename("B_biome").dropna().reset_index()

    if fallback_ko:
        print(f"  {biome}: biome ko_per_mb CSV missing — using ko_per_mb_primary as fallback")
        df_b = p1[["genus_lower", "ko_per_mb_primary", "phylum", "kingdom"]].merge(
            B_b, on="genus_lower", how="inner"
        ).rename(columns={"ko_per_mb_primary": "ko_per_mb_biome"})
    else:
        ko_df = pd.read_csv(ko_path)[["genus_lower", "ko_per_mb_biome"]].dropna()
        df_b = ko_df.merge(B_b, on="genus_lower", how="inner")
        df_b = df_b.merge(p1[["genus_lower", "phylum", "kingdom"]], on="genus_lower", how="inner")

    df_b["niche_z"] = z_score(df_b["B_biome"])
    df_b = df_b.dropna(subset=["niche_z", "ko_per_mb_biome"])
    print(f"  {biome}: n={len(df_b)}  fallback_ko={fallback_ko}")

    if len(df_b) < 40:
        print(f"    too few genera ({len(df_b)}) — skip PGLS")
        continue

    res = run_pgls(df_b, str(TREE_BAC), response="ko_per_mb_biome",
                   predictors=["niche_z"], taxon_col="genus_lower",
                   label=f"D_{biome}", min_n=25)
    r = extract_pgls(res, f"D_{biome}: {biome} biome {'(biome ko)' if not fallback_ko else '(fallback ko)'}")
    r.update({
        "ci_lo": r["beta"] - 1.96 * r["SE"],
        "ci_hi": r["beta"] + 1.96 * r["SE"],
        "sig": sig_label(r["p"]),
        "layer": f"D_{biome}",
        "data_type": dtype,
        "biome": biome,
    })
    biome_results.append(r)
    print(f"    beta={r['beta']:+.4f}  p={r['p']:.3e}  n={r['n']}  {r['sig']}")

biome_df = pd.DataFrame(biome_results)
biome_df.to_csv(DATA / "38_biome_stratified_pgls.csv", index=False)
print(f"\nSaved {len(biome_df)} biome-stratified results → 38_biome_stratified_pgls.csv")
_comp_results.extend(biome_results)


  soil: biome ko_per_mb CSV missing — using ko_per_mb_primary as fallback
  soil: n=0  fallback_ko=True
    too few genera (0) — skip PGLS
  marine: biome ko_per_mb CSV missing — using ko_per_mb_primary as fallback
  marine: n=0  fallback_ko=True
    too few genera (0) — skip PGLS


  freshwater: biome ko_per_mb CSV missing — using ko_per_mb_primary as fallback
  freshwater: n=1569  fallback_ko=True


    beta=-0.0692  p=3.653e-01  n=1569  ns

Saved 1 biome-stratified results → 38_biome_stratified_pgls.csv


## Section F: External Datasets

Checks for cached files and prints download instructions. Priority order: EPA Ecoregions > GLWD wetland > NADP atmospheric deposition > N deposition global.

In [18]:
# ── F: External Datasets ─────────────────────────────────────────────────────
# Each section checks for cached files and prints download instructions if missing.
import os

# ── F1: EPA Level III Ecoregions (USA, 86 regions) ──────────────────────────
ECO_GRID   = DATA / "epa_ecoregions_0.25deg.parquet"
ECO_SHP    = DATA / "epa_ecoregions"
ECO_COUNTS = DATA / "38_genus_ecoregion_counts.parquet"

if os.path.exists(ECO_COUNTS):
    r_f1 = parquet_levins_pgls(str(ECO_COUNTS), "eco_l3",
                                "F1: EPA Level III Ecoregion (USA)", "F1_ecoregion", "Habitat")
    if r_f1: _comp_results.append(r_f1)
elif os.path.exists(ECO_GRID):
    print("EPA Ecoregion grid available — run on cluster to join to MicrobeAtlas samples")
else:
    print("F1 EPA Ecoregions: not found.")
    print("  Download: wget https://gaftp.epa.gov/EPADataCommons/ORD/Ecoregions/us/us_eco_l3.zip")
    print(f"  Extract to: {ECO_SHP}")
    print("  Then rasterize with geopandas (0.25° grid) and save to:", ECO_GRID)
    print("  Then join to MicrobeAtlas genus occurrence in Spark (same pattern as C1)")

# ── F2: GLWD Wetland / Hydric Soils Indicator ────────────────────────────────
WETLAND_COUNTS = DATA / "38_genus_wetland_counts.parquet"
GLWD_PATH = Path("/home/hmacgregor/data/envdbs/glwd_wetland_fraction.parquet")

if os.path.exists(WETLAND_COUNTS):
    r_f2 = parquet_levins_pgls(str(WETLAND_COUNTS), "wetland_class",
                                "F2: Wetland indicator (GLWD)", "F2_wetland", "Habitat")
    if r_f2: _comp_results.append(r_f2)
elif os.path.exists(GLWD_PATH):
    print("GLWD parquet available — need Spark join to MicrobeAtlas occurrences")
    print("  Spark: join otu_counts_long × enriched_metadata × glwd grid, filter wetland_frac > 0.10")
else:
    print("F2 GLWD Wetland: not found.")
    print("  Download GLWD from https://www.worldwildlife.org/pages/global-lakes-and-wetlands-database")
    print("  Aggregate 30-arc-second wetland fraction to 0.25° grid")
    print(f"  Save to: {GLWD_PATH}")

# ── F3: NADP Atmospheric Deposition (Pb, USA) ────────────────────────────────
NADP_CSV = DATA / "nadp_pb_deposition_0.25deg.csv"
if os.path.exists(NADP_CSV):
    nadp = pd.read_csv(NADP_CSV)
    print(f"NADP loaded: {len(nadp)} grid cells, cols: {list(nadp.columns)}")
    print("  Join to MicrobeAtlas in Spark for per-genus Pb deposition SD PGLS")
else:
    print("F3 NADP: not found.")
    print("  Download NTN Pb deposition raster from https://nadp.slh.wisc.edu/maps/")
    print("  Resample to 0.25° and save as CSV: lat, lon, pb_wet_dep_kg_ha_yr")
    print(f"  Target: {NADP_CSV}")

# ── F4: N Deposition (global) ────────────────────────────────────────────────
NDEP_CSV = DATA / "n_deposition_global_0.25deg.csv"
if os.path.exists(NDEP_CSV):
    ndep = pd.read_csv(NDEP_CSV)
    print(f"N deposition loaded: {len(ndep)} cells")
else:
    print("F4 N deposition: not found.")
    print("  Options:")
    print("  - USA: https://nadp.slh.wisc.edu/ (NTN + AMON combined)")
    print("  - Global: Dentener et al. 2006 or EMEP gridded products")
    print(f"  Save to: {NDEP_CSV} with cols: lat, lon, n_dep_kg_ha_yr")

print("\nExternal dataset section complete.")
print(f"Current _comp_results count: {len(_comp_results)}")


F1 EPA Ecoregions: not found.
  Download: wget https://gaftp.epa.gov/EPADataCommons/ORD/Ecoregions/us/us_eco_l3.zip
  Extract to: /home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology/data/epa_ecoregions
  Then rasterize with geopandas (0.25° grid) and save to: /home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology/data/epa_ecoregions_0.25deg.parquet
  Then join to MicrobeAtlas genus occurrence in Spark (same pattern as C1)
F2 GLWD Wetland: not found.
  Download GLWD from https://www.worldwildlife.org/pages/global-lakes-and-wetlands-database
  Aggregate 30-arc-second wetland fraction to 0.25° grid
  Save to: /home/hmacgregor/data/envdbs/glwd_wetland_fraction.parquet
F3 NADP: not found.
  Download NTN Pb deposition raster from https://nadp.slh.wisc.edu/maps/
  Resample to 0.25° and save as CSV: lat, lon, pb_wet_dep_kg_ha_yr
  Target: /home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology/data/nadp_pb_depositi

## Section G: Summary Figures

In [19]:
# ── G1: Assemble all results + Figure 2 — Grand categorical forest ───────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# NB37 reference results (hardcoded from completed run)
NB37_REF = [
    {"label": "L0a: MA Levins B (P1 reference)", "beta": -0.517, "SE": 0.084,
     "p": 1.3e-8, "n": 1574, "lam": 0.85,
     "ci_lo": -0.517 - 1.96*0.084, "ci_hi": -0.517 + 1.96*0.084,
     "sig": "***", "layer": "L0a_ref", "data_type": "Reference"},
    {"label": "L1: Edaphic SD (pH+temp, ref)",   "beta":  0.153, "SE": 0.079,
     "p": 0.054, "n": 1573, "lam": 0.85,
     "ci_lo":  0.153 - 1.96*0.079, "ci_hi":  0.153 + 1.96*0.079,
     "sig": "ns", "layer": "L1_ref", "data_type": "Reference"},
    {"label": "L4: NGSA geochem SD (ref)",        "beta":  0.042, "SE": 0.080,
     "p": 0.600, "n": 1530, "lam": 0.85,
     "ci_lo":  0.042 - 1.96*0.080, "ci_hi":  0.042 + 1.96*0.080,
     "sig": "ns", "layer": "L4_ref", "data_type": "Reference"},
]

all_results = pd.DataFrame(NB37_REF + _comp_results)
all_results["ci_lo"] = all_results.apply(
    lambda r: r["ci_lo"] if pd.notna(r.get("ci_lo")) else r["beta"] - 1.96 * r["SE"], axis=1
)
all_results["ci_hi"] = all_results.apply(
    lambda r: r["ci_hi"] if pd.notna(r.get("ci_hi")) else r["beta"] + 1.96 * r["SE"], axis=1
)
all_results.to_csv(DATA / "38_categorical_niche_comparison.csv", index=False)
print(f"Total results: {len(all_results)}")
print(all_results[["label", "beta", "p", "sig", "data_type"]].to_string(index=False))

# ── Figure 2: Grand categorical forest ────────────────────────────────────────
TYPE_COLORS = {
    "Reference":    "#888888",
    "Habitat":      PALETTE[0],
    "Climate":      PALETTE[1],
    "Geochemical":  PALETTE[2],
    "Edaphic":      PALETTE[3],
    "Anthropogenic":PALETTE[4],
    "Other":        PALETTE[5],
}

df_plot = all_results.dropna(subset=["beta", "SE"]).reset_index(drop=True)
nrows = len(df_plot)
fig_h = max(ROW_H, nrows * 0.38 + 1.0)

fig, ax = plt.subplots(figsize=(FIGW["2col"], fig_h))
grid_h(ax)

y = list(range(nrows))
bar_colors = [TYPE_COLORS.get(dt, "#888") for dt in df_plot["data_type"]]
xerr_lo = (df_plot["beta"] - df_plot["ci_lo"]).clip(lower=0)
xerr_hi = (df_plot["ci_hi"] - df_plot["beta"]).clip(lower=0)

ax.barh(
    y, df_plot["beta"],
    xerr=[xerr_lo, xerr_hi],
    color=bar_colors, edgecolor="k", linewidth=0.5,
    capsize=2, height=0.6,
    error_kw={"elinewidth": 0.8, "ecolor": "#444"},
)
ax.axvline(0, color="gray", lw=0.8, ls="--")
ax.set_yticks(y)
ax.set_yticklabels(df_plot["label"].tolist(), fontsize=7.5)
ax.set_xlabel("PGLS β (metal gene load ~ categorical niche breadth)", fontsize=9)
ax.set_ylabel("")

for i, (_, row) in enumerate(df_plot.iterrows()):
    s = row.get("sig", "ns")
    if isinstance(s, str) and s != "ns":
        ax.annotate(s, xy=(row["ci_hi"] + 0.005, i), fontsize=7, va="center")
    n_val = row.get("n", float("nan"))
    n_str = str(int(n_val)) if pd.notna(n_val) else "—"
    ax.annotate(f"n={n_str}",
                xy=(ax.get_xlim()[0] + 0.001, i - 0.38),
                fontsize=6.5, color="#808080")

legend_patches = [mpatches.Patch(color=v, label=k) for k, v in TYPE_COLORS.items()
                  if k in df_plot["data_type"].values]
ax.legend(handles=legend_patches, fontsize=7, loc="lower right", framealpha=0.85)

ax.set_title("Categorical niche breadth vs metal gene load — all layers (PGLS)", fontsize=10)
fig.suptitle("NB38: Categorical Niche Breadth — Grand Comparison", y=1.02,
             fontsize=11, fontweight="bold")
save(fig, FIGS / "fig_nb38_grand_forest")
print("Saved fig_nb38_grand_forest.pdf")


Total results: 10
                                       label      beta            p sig     data_type
             L0a: MA Levins B (P1 reference) -0.517000 1.300000e-08 ***     Reference
               L1: Edaphic SD (pH+temp, ref)  0.153000 5.400000e-02  ns     Reference
                   L4: NGSA geochem SD (ref)  0.042000 6.000000e-01  ns     Reference
              B1: Soil habitat cat. Levins B -0.308817 5.191491e-05 ***       Habitat
                   C2: Köppen-Geiger climate  0.190170 2.700633e-02   *       Climate
                      C3: Population density -0.032591 6.834238e-01  ns Anthropogenic
                          C5a: SoilGrids CEC  0.590062 2.013945e-13 ***       Edaphic
                         C5b: SoilGrids clay  0.075715 3.312391e-01  ns       Edaphic
       C4b: NGSA geochem niche (+Fe/Mn, USA)  0.037627 6.407780e-01  ns   Geochemical
D_freshwater: freshwater biome (fallback ko) -0.069223 3.653333e-01  ns Anthropogenic
Saved fig_nb38_grand_forest.pdf


In [20]:
# ── G2: Figure 3 — Biome stratification panel ────────────────────────────────
import matplotlib.pyplot as plt, os

BIOME_CSV = DATA / "38_biome_stratified_pgls.csv"
if not os.path.exists(BIOME_CSV):
    print("Biome-stratified results not yet available — run D3 on cluster first.")
else:
    try:
        biome_df2 = pd.read_csv(BIOME_CSV)
    except Exception:
        biome_df2 = pd.DataFrame()

    if len(biome_df2) == 0:
        print("38_biome_stratified_pgls.csv is empty — no biome PGLS ran successfully.")
        print("Run D1/D2 on JupyterHub cluster, download parquets, then re-run D3.")
    else:
        # Reference: L0a all-biome
        ref_beta  = -0.517
        ref_ci_lo = ref_beta - 1.96 * 0.084
        ref_ci_hi = ref_beta + 1.96 * 0.084

        fig, ax = plt.subplots(figsize=(FIGW["1.5col"], ROW_H))
        grid_h(ax)

        biomes = biome_df2["biome"].tolist()
        betas  = biome_df2["beta"].tolist()
        sigs   = biome_df2["sig"].tolist()
        cis_lo = biome_df2["ci_lo"].tolist()
        cis_hi = biome_df2["ci_hi"].tolist()
        ns_    = biome_df2["n"].tolist()

        y = list(range(len(biomes)))
        colors = [PALETTE[i % len(PALETTE)] for i in range(len(biomes))]
        xerr_lo = [b - lo for b, lo in zip(betas, cis_lo)]
        xerr_hi = [hi - b  for b, hi  in zip(betas, cis_hi)]

        ax.barh(y, betas, xerr=[xerr_lo, xerr_hi],
                color=colors, edgecolor="k", linewidth=0.5,
                capsize=3, height=0.6,
                error_kw={"elinewidth": 0.8, "ecolor": "#444"})
        ax.axvline(0, color="gray", lw=0.8, ls="--")
        ax.axvspan(ref_ci_lo, ref_ci_hi, alpha=0.12, color="gray",
                   label=f"L0a 95% CI (all-biome)")
        ax.axvline(ref_beta, color="gray", lw=1.2, ls=":",
                   label=f"L0a β = {ref_beta}")

        ax.set_yticks(y)
        ax.set_yticklabels([b.capitalize() for b in biomes], fontsize=8)
        ax.set_xlabel("PGLS β (biome-specific ko_per_mb ~ categorical Levins B)", fontsize=9)
        ax.set_ylabel("")

        xlim = ax.get_xlim()
        for i, (b, s, n, ci_hi) in enumerate(zip(betas, sigs, ns_, cis_hi)):
            if isinstance(s, str) and s != "ns":
                ax.annotate(s, xy=(ci_hi + 0.003, i), fontsize=7, va="center")
            n_str = str(int(n)) if pd.notna(n) else "—"
            ax.annotate(f"n={n_str}", xy=(xlim[0] + 0.001, i - 0.38), fontsize=7, color="#808080")

        ax.legend(fontsize=7, loc="lower right")
        ax.set_title("Biome-stratified categorical niche breadth PGLS", fontsize=10)
        fig.suptitle("NB38 D: Biome Stratification", y=1.02, fontsize=11, fontweight="bold")
        save(fig, FIGS / "fig_nb38_biome_stratified")
        print("Saved fig_nb38_biome_stratified.pdf")


Saved fig_nb38_biome_stratified.pdf
